In [3]:
import pandas as pd
from pathlib import Path
from tqdm import tqdm

# 输入输出路径
INPUT_DIR = Path("/workspace/Llama4AML_250721/explainability/flow_blocks_third_time")
OUTPUT_PATH = Path("/workspace/Llama4AML_250721/explainability/llama_prompts/money_laundering_prompts.json")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

def build_prompt(df: pd.DataFrame, file_name: str) -> str:
    df = df.sort_values("timeStamp")
    steps = []

    for _, row in df.iterrows():
        try:
            tx_type = row['transaction_type']
            hash_id = str(row['hash'])[:10]
            from_addr = str(row['from'])[:10]
            to_addr = str(row['to'])[:10]
            timestamp = row['timestamp']
            value = str(row['value'])
            token = row.get('tokenSymbol', 'Unknown')
            contract = str(row.get('contractAddress', ''))[:10]
            gasFee = row.get('gasFee', 'Unknown')

            if tx_type in ['direct_eth_transfer', 'direct_token_transfer']:
                step = (
                    f"In transaction {hash_id}, account {from_addr} sent "
                    f"{value} {token} to account {to_addr} at time {timestamp}, with {gasFee} gas fee."
                )

            elif tx_type == 'token_swap':
                swap_amount = row.get('swap_amount', 'Unknown')
                swap_token_symbol = row.get('swap_token_name', 'Unknown')
                step = (
                    f"In transaction {hash_id}, account {from_addr} used account {to_addr} to swap "
                    f"{value} {token} into {swap_amount} {swap_token_symbol} at time {timestamp}, with {gasFee} gas fee."
                )

            elif tx_type == 'indirect_transfer':
                transfer_to_true = row.get('transfer_to_true', 'Unknown')
                step = (
                    f"In transaction {hash_id}, account {from_addr} used account {to_addr} to transfer "
                    f"{value} {token} to account {transfer_to_true} at {timestamp}, with {gasFee} gas fee."
                )

            elif tx_type == 'cross_chain':
                blockchain = row.get('cross_to_blockchain', 'unknown')
                step = (
                    f"In transaction {hash_id}, account {from_addr} used account {to_addr} to "
                    f"cross-chain transfer {value} {token} at {blockchain} blockchain at {timestamp}, with  {gasFee} gas fee."
                )

            else:

                step = (
                    f"In transaction {hash_id}, account {from_addr} sent {value} {token} ({contract}) "
                    f"to account {to_addr} at time {timestamp}, with {gasFee} gas fee."
                )

            steps.append(step)

        except Exception as e:
            steps.append(f"[Error parsing transaction: {e}]")

    # 拼接整体 prompt
    flow_description = "\n".join([f"{i+1}. {s}" for i, s in enumerate(steps)])
    # prompt = (
    #     f"This is a suspicious money laundering cryptocurrency transaction flow extracted from `{file_name}`. "
    #     f"The flow contains the following steps:\n\n{flow_description}\n\n"
    #     "Please analyze the above transaction flow and provide the following information:\n\n"
    #     "1. **Money Laundering Path**: Describe how the funds are transferred step by step. "
    #     "Include address abbreviations, amounts, token names, and time. Clearly outline the direction and transformation of the assets.\n"
    #     "2. **Money Laundering Techniques Used**: Analyze the techniques used in this flow, such as layering, mixing, cross-chain bridging, structuring, or using intermediaries and so on.\n"
    #     "3. **Suspicious Behaviors or Red Flags**: Identify abnormal behaviors in this flow. \n\n"
    #     "Please format your response in a structured and concise manner."
    # )
    prompt = f"""
    You are a professional blockchain forensic analyst specializing in Ethereum-based money laundering detection.

    The following is a suspicious transaction flow extracted from `{file_name}`.

    === Transaction Flow ===
    {flow_description}
    ========================


    Provide a **high-level forensic analysis** in **structured Markdown format**, focusing on **summary**, **techniques**, and **red flags**.

    Your report should contain the following sections:

    ---

    ### 1. 🧭 Money Laundering Path  
    - Provide a summarized narrative of how the assets flowed through the network.
    - Describe major phases (e.g., initial injection, conversion, routing, distribution).
    - Explain how the flow structure supports laundering objectives (e.g., fragmentation, obfuscation, integration).
    - Estimate total amount laundered if possible (approximate is fine).

    ---

    ### 2. 🧪 Money Laundering Techniques Used  
    Identify and explain any techniques used in this flow, such as:
    - Layering, structuring, mixing, contract routing, timing tricks, address reuse, etc.
    - Don't limit yourself to this list—feel free to include novel or uncommon tactics if observed.
    - Mention if techniques were combined to increase obfuscation.

    ---

    ### 3. 🚩 Suspicious Behaviors and Red Flags  
    List and explain behavioral anomalies or risk indicators, such as:
    - Rapid fund movement, repeated swap patterns, identical chunk sizes, abnormal gas use, contract reuse, centralized redistribution, and so on.
    - Include both structural and temporal abnormalities.
    - Highlight any account behavior suggesting automation or laundering coordination.

    ---

    Please format your output in Markdown. Be analytical, concise, and forensic in tone.
    """

    # prompt = f"""
    # You are a forensic blockchain analyst specialized in detecting money laundering behavior in Ethereum.

    # The following is a suspicious transaction flow extracted from `{file_name}`.

    # === Transaction Steps ===
    # {flow_description}
    # =========================

    # Please answer the following questions **in structured markdown format**:

    # ### 1. Money Laundering Path
    # - Step-by-step flow summary (include amounts, token types, sender/receiver abbreviation).
    # - Total assets involved and their transformation process.

    # ### 2. Money Laundering Techniques Used
    # Identify which laundering techniques appear in this flow (e.g., layering, mixing, structuring, cross-chain, use of intermediaries). Explain briefly.

    # ### 3. Suspicious Behaviors or Red Flags
    # List and explain red flags in this transaction flow (e.g., unusual volume splits, fast sequences, repeated patterns, self-interactions, gas pattern anomalies,and so on).

    # Format your response using Markdown with bullet points or tables. Be concise and analytical.
    # """

    return prompt


In [4]:
import json
from pathlib import Path
import pandas as pd
from tqdm import tqdm

all_prompts = []

# 遍历处理所有 csv 文件
for csv_file in tqdm(sorted(INPUT_DIR.glob("*.csv"))):
    try:
        df = pd.read_csv(csv_file)
        if len(df) <= 1:
            continue
        prompt = build_prompt(df, csv_file.name)
        all_prompts.append({
            "file": csv_file.name,
            "prompt": prompt
        })
    except Exception as e:
        print(f"[!] Error in {csv_file.name}: {e}")

# 写入 JSON 文件（而不是 JSONL）
with open(OUTPUT_PATH, "w") as fout:
    json.dump(all_prompts, fout, indent=2)

100%|██████████| 57/57 [00:00<00:00, 234.66it/s]
